> 在线讲义：[第 13 章 · 整个模型的资源需求与比较：存多少、算多少、能放几条请求](https://wikieden.github.io/ai-infra-from-constraints/lessons/13-model-comparison.html) · 本书《从约束推导 AI Infra》

# 第 13 章 · 整个模型的资源需求与比较：存多少、算多少、能放几条请求

配套讲义：仓库里 `ai-infra/notes/lessons/13-model-comparison.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 路由专家占比 | 专家 ÷ 总和 | `expert_share(name)` | Qwen3.6 93%，V4-Flash 97%，Kimi K3 98% |
| 第二步 完整权重 | 总参数 × 2 | `bf16_gb(name)` | 16.38、69.3、568.7 GB…… |
| 第三步 RTX 4090 | $\lfloor \cdot \rfloor$ | `b_max(24e9, QWEN8B_W, 2 GiB, KV)` | 4K 9 条、8K 4 条、16K 2 条 |
| 第三步 70B 量化 | 同上 | `b_max(80e9, 39.5e9, …)` | 8 比特 1 条，4 比特 14 条，32K 3 条 |
| 第三步 反推 | $N_{\max}$ | `n_max(24e9, 2 GiB, 4, 1.125 GiB, 2)` | 约 85.1 亿 |
| 第四步 增长倍数 | 1M ÷ 8K | `growth()` | Qwen3-8B 128 倍，V4.1 约 6.9 倍 |

表 2-B 参数分解、表 2-E 长上下文交互、RTX 4090 与 H100 的容量算例取自 AII 第 2 章 §2.6。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
GIB = 2**30
PARAMS = {
    "V4.1 Flash": (1.324, 5.126, 1.416, 543.582, 196.929, 0.118),
    "Qwen3-8B": (1.245, 1.510, 5.436, 0.0, 0.0, 0.0),
    "Qwen3.6": (1.017, 1.283, 0.126, 32.212, 0.0, 0.022),
    "V4-Flash": (1.059, 5.085, 1.082, 277.025, 0.0, 0.081),
    "Kimi K3": (2.349, 36.180, 12.882, 2722.741, 0.0, 5.333),
}
LONG = {
    "V4.1 Flash": (3.66, 25.23, 0.029, 3.138),
    "Qwen3-8B": (4.83, 618.48, 1.125, 144.000),
    "Qwen3.6": (1.34, 171.80, 0.217, 20.060),
    "V4-Flash": (3.00, 113.80, 0.069, 6.735),
    "Kimi K3": (41.08, 5257.04, 0.634, 27.423),
}
QWEN8B_W = 16_381_470_720  # Qwen3-8B BF16 权重字节

**函数 `expert_share`**：路由专家占全部参数的比例。

In [ ]:
def expert_share(name: str) -> float:
    """路由专家占全部参数的比例。"""
    p = PARAMS[name]
    return p[3] / sum(p)

**函数 `bf16_gb`**：统一 BF16 时的完整权重（GB）= 总参数（十亿）× 2。

In [ ]:
def bf16_gb(name: str) -> float:
    """统一 BF16 时的完整权重（GB）= 总参数（十亿）× 2。"""
    return sum(PARAMS[name]) * 2

**函数 `b_max`**：一张卡能同时放几条请求：(容量 - 权重 - 工作区) ÷ 每请求状态，向下取整。

In [ ]:
def b_max(device: float, weight: float, workspace: float, per_request: float) -> int:
    """一张卡能同时放几条请求：(容量 - 权重 - 工作区) ÷ 每请求状态，向下取整。"""
    return int((device - weight - workspace) // per_request)

**函数 `n_max`**：反推权重上限（参数个数）：(C - M0 - B·K) / b_W。

In [ ]:
def n_max(cap: float, workspace: float, B: int, kv: float, bytes_per_param: float) -> float:
    """反推权重上限（参数个数）：(C - M0 - B·K) / b_W。"""
    return (cap - workspace - B * kv) / bytes_per_param

**函数 `growth`**：8K → 1M 时交互运算涨了几倍。

In [ ]:
def growth() -> dict[str, float]:
    """8K → 1M 时交互运算涨了几倍。"""
    return {k: v[1] / v[0] for k, v in LONG.items()}

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：路由专家占参数的比例")
for name in ("Qwen3.6", "V4-Flash", "Kimi K3"):
    print(f"  {name:10s} {expert_share(name):.1%}")

In [ ]:
print("\n第二步：统一 BF16 的完整权重")
for name in PARAMS:
    print(f"  {name:10s} {bf16_gb(name):8.1f} GB")

In [ ]:
print("\n第三步：RTX 4090（24 GB），Qwen3-8B BF16，工作区 2 GiB")
for ctx, kv in ((4096, 0.5625), (8192, 1.125), (16384, 2.25)):
    print(f"  {ctx // 1024}K 上下文（KV {kv} GiB）→ {b_max(24e9, QWEN8B_W, 2 * GIB, kv * GIB)} 条")
print("  70B 在 H100（80 GB），工作区 2 GiB，8K 请求 KV 2.5 GiB：")
print(f"    8 比特 73.726 GB → {b_max(80e9, 73.726e9, 2 * GIB, 2.5 * GIB)} 条")
print(f"    4 比特 39.500 GB → {b_max(80e9, 39.5e9, 2 * GIB, 2.5 * GIB)} 条；32K（KV 10 GiB）→ {b_max(80e9, 39.5e9, 2 * GIB, 10 * GIB)} 条")
print(f"  反推：4090 放 4 条 8K 请求后，BF16 最多 {n_max(24e9, 2 * GIB, 4, 1.125 * GIB, 2) / 1e9:.2f} 十亿参数")

In [ ]:
print("\n第四步：8K → 1M，交互运算涨了几倍")
for name, g in growth().items():
    print(f"  {name:10s} {g:6.1f} 倍")

## 3. 代码里才看得到的

1. 显存用十进制 GB（24e9），工作区和 KV 用 GiB（$2^{30}$）：混算前统一成字节；把 GiB 当 GB 用，每请求 KV 会少算约 7%，在临界处就会多算出一条请求。
2. 表 2-B 的数是四舍五入后的十亿，加起来的总参数和表 2-A 有千分之几的出入；`bf16_gb` 用表 2-B 相加。
3. `b_max` 没算每请求的递推和卷积状态（Qwen3-8B 没有）；混合模型要把它们一起放进分母（原文 L813）。

## 4. 自己改一改

- 把 4090 换成 48 GB 的卡，看 8K 能放几条 Qwen3-8B 请求。
- 把 70B 4-bit 的上下文改成 16K，看能放几条。
- 用 `growth()` 的数据算一算：V4-Flash 从 8K 到 1M，状态涨了多少倍？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 存、算、账本三本账，各算各的别混一谈；
> 专家占了九成七，单 token 只算挑中的；
> 停车场先划固定位，剩下除以每辆车，向下取整；
> 一千页的书，从头翻还是看目录，涨得天差地别。